In [ ]:
!pip install -q librosa soundfile transformers catboost scikit-learn tqdm
print('✅ Dependencies installed')

In [ ]:
# ── DIAGNOSTIC: Run this to find your correct paths ──────────────────────────
import os

search_roots = ['/kaggle/input', '/content', '.', '..']

print("=== Searching for dataset files ===\n")
for root in search_roots:
    if not os.path.exists(root):
        continue
    for dirpath, dirnames, filenames in os.walk(root):
        if any(g in dirnames for g in ['blues', 'genres_stems', 'mashups', 'ESC-50-master']):
            print(f"FOUND at: {dirpath}")
            print(f"  Contents: {sorted(dirnames[:10])}\n")
        depth = dirpath.replace(root, '').count(os.sep)
        if depth >= 4:
            dirnames.clear()

candidates = [
    '/kaggle/input/messy-mashup/genres_stems',
    '/kaggle/input/messy-mashup/messy_mashup/genres_stems',
    '/kaggle/input/genres_stems',
]
for c in candidates:
    e = os.path.exists(c)
    print(f"{'✅' if e else '❌'}  {c}")
    if e:
        print(f"   → {sorted(os.listdir(c))}")

In [ ]:
import os, random, warnings
import numpy as np
import torch
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
STEMS_DIR   = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems'          # genre/song/stem.wav
NOISE_DIR   = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio'   # .wav noise files
MASHUPS_DIR = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups'               # test audio files
TEST_CSV    = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv'

DATASET_DIR   = './mashup_dataset'    # generated .pt tensors
CKPT_DIR      = './checkpoints'
SUBMISSION    = './submission.csv'

# ── Audio config ───────────────────────────────────────────────────────────
GENRES = ['blues','classical','country','disco','hiphop',
          'jazz','metal','pop','reggae','rock']
STEMS  = ['drums.wav','vocals.wav','bass.wav','others.wav']
NUM_CLASSES   = 10
SAMPLE_RATE   = 22050
CLIP_DURATION = 10       # seconds
N_MELS        = 128
N_FFT         = 2048
HOP_LENGTH    = 512
TARGET_LEN    = 431      # mel time-frames for 10 s at sr=22050, hop=512

# ── Training config ────────────────────────────────────────────────────────
NUM_SAMPLES   = 30000    # mashup samples to generate
RESNET_EPOCHS = 50
AST_EPOCHS    = 10
BATCH_RESNET  = 256
BATCH_AST     = 32
BATCH_EFFNET     = 128
N_TTA         = 10       # test-time augmentation views
SEED          = 42
USE_MIXUP        = True    # NEW
MIXUP_ALPHA      = 0.4  

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

LABEL_MAP   = {g: i for i, g in enumerate(GENRES)}
IDX_TO_GENRE = {i: g for g, i in LABEL_MAP.items()}

print(f'Device : {DEVICE}')
print(f'Genres : {GENRES}')
print(f'Label  : {LABEL_MAP}')

In [ ]:
# Pre-warm numba so librosa imports faster
import numba
print(numba.__version__)

In [ ]:
import os
os.environ['NUMBA_CACHE_DIR'] = '/tmp/numba_cache'   # cache JIT across runs
os.environ['NUMBA_DISABLE_JIT'] = '0'

import librosa
import librosa.effects

def load_audio(path, duration=None):
    try:
        y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True, duration=duration)
        return y.astype(np.float32)
    except Exception:
        n = int(SAMPLE_RATE * (duration or CLIP_DURATION))
        return np.zeros(n, dtype=np.float32)


def time_stretch_safe(y, rate):
    try:
        return librosa.effects.time_stretch(y, rate=rate)
    except Exception:
        return y


def pitch_shift_safe(y, n_steps):
    """NEW: pitch shift by n_steps semitones."""
    try:
        return librosa.effects.pitch_shift(y, sr=SAMPLE_RATE, n_steps=n_steps)
    except Exception:
        return y


# ── Global normalization stats (computed from training set after generation) ──
# Will be filled in after dataset generation
GLOBAL_MEL_MEAN = None
GLOBAL_MEL_STD  = None


def audio_to_mel(y, global_norm=False):
    """
    Waveform → log-mel tensor (1, N_MELS, TARGET_LEN).
    Uses per-sample z-score during training generation.
    Uses global stats during inference (if global_norm=True).
    """
    mel = librosa.feature.melspectrogram(
        y=y, sr=SAMPLE_RATE, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
    log_mel = librosa.power_to_db(mel, ref=np.max).astype(np.float32)

    # Pad / crop time axis
    if log_mel.shape[1] < TARGET_LEN:
        log_mel = np.pad(log_mel, ((0,0),(0, TARGET_LEN - log_mel.shape[1])),
                         constant_values=-80.0)
    else:
        log_mel = log_mel[:, :TARGET_LEN]

    if global_norm and GLOBAL_MEL_MEAN is not None:
        log_mel = (log_mel - GLOBAL_MEL_MEAN) / (GLOBAL_MEL_STD + 1e-6)
    else:
        # Per-sample z-score (robust to loudness differences)
        log_mel = (log_mel - log_mel.mean()) / (log_mel.std() + 1e-6)

    return torch.tensor(log_mel, dtype=torch.float32).unsqueeze(0)  # (1,128,431)


print('✅ Audio utilities ready')

In [ ]:
from pathlib import Path

NOISE_FILES = list(Path(NOISE_DIR).rglob('*.wav'))
print(f'Found {len(NOISE_FILES)} ESC-50 noise files')


def mix_stems(genre):
    """
    UPGRADED: Cross-song stem mixing with pitch augmentation.
    - Pick up to 4 different songs
    - Random tempo + pitch jitter
    - Occasionally mix in a stem from a DIFFERENT genre (hard negative)
    """
    target_len = int(SAMPLE_RATE * CLIP_DURATION)
    song_dirs  = sorted(p for p in (Path(STEMS_DIR) / genre).iterdir() if p.is_dir())
    if len(song_dirs) < 2:
        song_dirs = song_dirs * 4

    chosen_songs = random.sample(song_dirs, min(4, len(song_dirs)))
    stem_names   = random.sample(STEMS, len(chosen_songs))
    mix = np.zeros(target_len, dtype=np.float32)

    for song_dir, stem_name in zip(chosen_songs, stem_names):
        stem_path = song_dir / stem_name
        if not stem_path.exists():
            continue
        y = load_audio(str(stem_path), duration=CLIP_DURATION + 2)

        # Tempo jitter
        if random.random() < 0.5:
            y = time_stretch_safe(y, random.uniform(0.85, 1.15))

        # NEW: pitch shift ±3 semitones
        if random.random() < 0.4:
            n_steps = random.uniform(-3, 3)
            y = pitch_shift_safe(y, n_steps)

        if len(y) >= target_len:
            start = random.randint(0, len(y) - target_len)
            y = y[start:start + target_len]
        else:
            y = np.pad(y, (0, target_len - len(y)))

        mix += random.uniform(0.15, 0.7) * y

    # NEW: occasionally add a single stem from a different genre (10% chance)
    if random.random() < 0.10:
        other_genre  = random.choice([g for g in GENRES if g != genre])
        other_songs  = sorted(p for p in (Path(STEMS_DIR) / other_genre).iterdir() if p.is_dir())
        if other_songs:
            od = random.choice(other_songs)
            sn = random.choice(STEMS)
            sp = od / sn
            if sp.exists():
                y2 = load_audio(str(sp), duration=CLIP_DURATION + 2)
                if len(y2) >= target_len:
                    y2 = y2[:target_len]
                else:
                    y2 = np.pad(y2, (0, target_len - len(y2)))
                mix += random.uniform(0.05, 0.15) * y2  # low gain → subtle contamination

    peak = np.abs(mix).max()
    if peak > 1e-6:
        mix /= peak
    return mix


def add_esc50_noise(y):
    """
    UPGRADED: wider SNR range (3–25 dB, was 5–30) → harder/more realistic.
    Also adds random room IR convolution (simple FIR approximation).
    """
    target_len = len(y)
    n_clips = random.randint(1, min(4, len(NOISE_FILES)))  # up to 4 noise clips
    for nf in random.sample(NOISE_FILES, n_clips):
        noise = load_audio(str(nf), duration=CLIP_DURATION + 2)
        if len(noise) == 0:
            continue
        clip_len = random.randint(int(SAMPLE_RATE), int(SAMPLE_RATE * 4))
        clip_len = min(clip_len, len(noise))
        ns  = random.randint(0, max(0, len(noise) - clip_len))
        nc  = noise[ns: ns + clip_len]
        pos = random.randint(0, max(0, target_len - clip_len))
        end = min(pos + len(nc), target_len)
        nc  = nc[:end - pos]

        snr_db     = random.uniform(3, 25)  # ← wider / harder
        sig_rms    = np.sqrt(np.mean(y ** 2)) + 1e-8
        noise_rms  = np.sqrt(np.mean(nc ** 2)) + 1e-8
        alpha      = sig_rms / (noise_rms * (10 ** (snr_db / 20)))
        y[pos:end] += alpha * nc

    peak = np.abs(y).max()
    if peak > 1e-6:
        y /= peak
    return y.astype(np.float32)


print('✅ Mashup generator ready')

In [ ]:
from tqdm.notebook import tqdm

samples_per_genre = NUM_SAMPLES // len(GENRES)
total_saved = 0

for genre in GENRES:
    genre_out = Path(DATASET_DIR) / genre
    genre_out.mkdir(parents=True, exist_ok=True)
    label = LABEL_MAP[genre]

    for i in tqdm(range(samples_per_genre), desc=f'{genre}', leave=False):
        y   = mix_stems(genre)
        y   = add_esc50_noise(y)
        mel = audio_to_mel(y)
        out = genre_out / f'{i:05d}.pt'
        torch.save({'mel': mel, 'label': label}, out)
        total_saved += 1

print(f'\n✅ Generated {total_saved:,} samples across {len(GENRES)} genres')

In [ ]:
# ── Compute global mel stats from a random subset of training data ──────────
# This normalises test inference consistently with training
print('Computing global mel mean/std from training data...')
all_pts = list(Path(DATASET_DIR).rglob('*.pt'))
sample_pts = random.sample(all_pts, min(3000, len(all_pts)))

running_sum  = 0.0
running_sq   = 0.0
count        = 0

for pt in tqdm(sample_pts, desc='Stats'):
    mel = torch.load(pt, map_location='cpu')['mel'].numpy()  # (1,128,431)
    running_sum += mel.mean()
    running_sq  += (mel ** 2).mean()
    count += 1

GLOBAL_MEL_MEAN = running_sum / count
GLOBAL_MEL_STD  = np.sqrt(running_sq / count - GLOBAL_MEL_MEAN ** 2)
print(f'Global mel  mean={GLOBAL_MEL_MEAN:.4f}  std={GLOBAL_MEL_STD:.4f}')
print('✅ Global stats ready')

In [ ]:
from torch.utils.data import Dataset, DataLoader


class MelDataset(Dataset):
    def __init__(self, file_paths, augment=False):
        self.paths   = file_paths
        self.augment = augment

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        data  = torch.load(self.paths[idx], map_location='cpu')
        mel   = data['mel']
        label = data['label']
        if self.augment:
            mel = self._spec_augment_v2(mel)
        return mel, label

    @staticmethod
    def _spec_augment_v2(mel, fmask=27, tmask=80, n_freq=2, n_time=2):
        """SpecAugment with stronger masks (was fmask=20, tmask=60)."""
        mel = mel.clone()
        _, F, T = mel.shape
        for _ in range(n_freq):
            f  = random.randint(0, fmask)
            f0 = random.randint(0, max(0, F - f - 1))
            mel[:, f0:f0+f, :] = 0
        for _ in range(n_time):
            t  = random.randint(0, tmask)
            t0 = random.randint(0, max(0, T - t - 1))
            mel[:, :, t0:t0+t] = 0
        return mel


def make_split(data_dir, val_frac=0.1, seed=SEED):
    all_files = []
    for genre in GENRES:
        all_files += [str(p) for p in (Path(data_dir)/genre).glob('*.pt')]
    rng = random.Random(seed)
    rng.shuffle(all_files)
    val_n = int(val_frac * len(all_files))
    return all_files[val_n:], all_files[:val_n]


def build_loaders_from_split(train_files, val_files,
                              batch_train=256, batch_val=256):
    train_ds = MelDataset(train_files, augment=True)
    val_ds   = MelDataset(val_files,   augment=False)
    train_ld = DataLoader(train_ds, batch_size=batch_train,
                          shuffle=True,  num_workers=2, pin_memory=True)
    val_ld   = DataLoader(val_ds,   batch_size=batch_val,
                          shuffle=False, num_workers=2, pin_memory=True)
    print(f'Train: {len(train_ds):,}  Val: {len(val_ds):,}')
    return train_ld, val_ld


TRAIN_FILES, VAL_FILES = make_split(DATASET_DIR)
print(f'Shared split → Train: {len(TRAIN_FILES):,}  Val: {len(VAL_FILES):,}')
print('✅ Dataset class ready')

In [ ]:
import torch.nn as nn
from sklearn.metrics import f1_score


def mixup_data(x, y, alpha=MIXUP_ALPHA):
    """NEW: Mixup augmentation for better boundary learning."""
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    y_a, y_b = y, y[idx]
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


def train_one_epoch(model, loader, optimizer, criterion, scaler,
                    use_mixup=False):
    model.train()
    total_loss = total_correct = total = 0
    for mel, labels in tqdm(loader, leave=False, desc='  train'):
        mel, labels = mel.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()

        if use_mixup and random.random() < 0.5:
            mel, y_a, y_b, lam = mixup_data(mel, labels)
            with torch.cuda.amp.autocast(enabled=(scaler is not None)):
                logits = model(mel)
                loss   = mixup_criterion(criterion, logits, y_a, y_b, lam)
        else:
            with torch.cuda.amp.autocast(enabled=(scaler is not None)):
                logits = model(mel)
                loss   = criterion(logits, labels)

        if scaler:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        total_loss    += loss.item() * len(labels)
        total_correct += (logits.argmax(1) == labels).sum().item()
        total         += len(labels)
    return total_loss / total, total_correct / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels, logits_all = [], [], []
    for mel, lbl in tqdm(loader, leave=False, desc='  eval '):
        mel = mel.to(DEVICE)
        out = model(mel)
        preds.extend(out.argmax(1).cpu().numpy())
        labels.extend(lbl.numpy())
        logits_all.append(out.cpu())
    f1  = f1_score(labels, preds, average='macro')
    return f1, torch.cat(logits_all), np.array(labels)


print('✅ Training helpers ready')

In [ ]:
from torchvision import models
import timm


class ResNetMel(nn.Module):
    """
    UPGRADED: ResNet50 (was ResNet18) — deeper features.
    """
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

        old = backbone.conv1
        new = nn.Conv2d(1, old.out_channels,
                        kernel_size=old.kernel_size,
                        stride=old.stride,
                        padding=old.padding, bias=False)
        with torch.no_grad():
            new.weight = nn.Parameter(old.weight.mean(dim=1, keepdim=True))
        backbone.conv1 = new

        backbone.fc = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(backbone.fc.in_features, num_classes)
        )
        self.net = backbone

    def forward(self, x):
        return self.net(x)


class EfficientNetMel(nn.Module):
    """
    NEW: EfficientNet-B2 for complementary feature extraction.
    """
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b2', pretrained=True, num_classes=num_classes,
            in_chans=1, drop_rate=0.3, drop_path_rate=0.2
        )

    def forward(self, x):
        return self.backbone(x)


# Sanity checks
with torch.no_grad():
    dummy = torch.randn(2, 1, N_MELS, TARGET_LEN)
    out_r = ResNetMel()(dummy)
    assert out_r.shape == (2, NUM_CLASSES)
    out_e = EfficientNetMel()(dummy)
    assert out_e.shape == (2, NUM_CLASSES)
print(f'✅ ResNet50 output shape    : {out_r.shape}')
print(f'✅ EfficientNet-B2 output   : {out_e.shape}')

In [ ]:
# ── ResNet50 Training ───────────────────────────────────────────────────────
train_ld, val_ld = build_loaders_from_split(TRAIN_FILES, VAL_FILES,
                                             batch_train=BATCH_RESNET)

resnet      = ResNetMel().to(DEVICE)
criterion_r = nn.CrossEntropyLoss(label_smoothing=0.1)

# NEW: 3-phase LR — warmup for 3 epochs, then cosine
optimizer_r = torch.optim.AdamW(resnet.parameters(), lr=3e-4, weight_decay=1e-4)

def get_scheduler(optimizer, warmup_epochs, total_epochs):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return float(epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return 0.5 * (1 + np.cos(np.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

scheduler_r = get_scheduler(optimizer_r, warmup_epochs=3, total_epochs=RESNET_EPOCHS)
scaler_r    = torch.cuda.amp.GradScaler() if DEVICE.type == 'cuda' else None

best_f1_r         = 0.0
oof_logits_resnet = None
oof_labels_resnet = None

for epoch in range(1, RESNET_EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(resnet, train_ld, optimizer_r,
                                      criterion_r, scaler_r,
                                      use_mixup=USE_MIXUP)
    val_f1, val_logits, val_labels = evaluate(resnet, val_ld)
    scheduler_r.step()

    marker = ''
    if val_f1 > best_f1_r:
        best_f1_r         = val_f1
        oof_logits_resnet = val_logits
        oof_labels_resnet = val_labels
        torch.save({'model_state': resnet.state_dict(), 'val_f1': val_f1},
                   f'{CKPT_DIR}/best_resnet50.pt')
        marker = '  ← best'

    print(f'[ResNet50] Ep {epoch:02d}/{RESNET_EPOCHS}  '
          f'loss={tr_loss:.4f}  acc={tr_acc:.4f}  '
          f'val_f1={val_f1:.4f}{marker}')

print(f'\n✅ ResNet50 best val macro-F1: {best_f1_r:.4f}')

In [ ]:
# ── EfficientNet-B2 Training (NEW) ─────────────────────────────────────────
EFFNET_EPOCHS = 40  # add this line

train_ld_e, val_ld_e = build_loaders_from_split(TRAIN_FILES, VAL_FILES,
                                                  batch_train=BATCH_EFFNET)

effnet      = EfficientNetMel().to(DEVICE)
criterion_e = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer_e = torch.optim.AdamW(effnet.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler_e = get_scheduler(optimizer_e, warmup_epochs=3, total_epochs=EFFNET_EPOCHS)
scaler_e    = torch.cuda.amp.GradScaler() if DEVICE.type == 'cuda' else None

best_f1_e         = 0.0
oof_logits_effnet = None
oof_labels_effnet = None

for epoch in range(1, EFFNET_EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(effnet, train_ld_e, optimizer_e,
                                      criterion_e, scaler_e,
                                      use_mixup=USE_MIXUP)
    val_f1, val_logits, val_labels = evaluate(effnet, val_ld_e)
    scheduler_e.step()

    marker = ''
    if val_f1 > best_f1_e:
        best_f1_e         = val_f1
        oof_logits_effnet = val_logits
        oof_labels_effnet = val_labels
        torch.save({'model_state': effnet.state_dict(), 'val_f1': val_f1},
                   f'{CKPT_DIR}/best_effnet_b2.pt')
        marker = '  ← best'

    print(f'[EffNet-B2] Ep {epoch:02d}/{EFFNET_EPOCHS}  '
          f'loss={tr_loss:.4f}  acc={tr_acc:.4f}  '
          f'val_f1={val_f1:.4f}{marker}')

print(f'\n✅ EfficientNet-B2 best val macro-F1: {best_f1_e:.4f}')

In [ ]:
from transformers import ASTForAudioClassification

AST_HF = 'MIT/ast-finetuned-audioset-10-10-0.4593'


class ASTGenreClassifier(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.cnn_stem = nn.Sequential(
            nn.Conv2d(1,  64,  3, 2, 1), nn.BatchNorm2d(64),  nn.GELU(),
            nn.Conv2d(64, 128, 3, 2, 1), nn.BatchNorm2d(128), nn.GELU(),
            nn.Conv2d(128,768, 3, 2, 1), nn.BatchNorm2d(768), nn.GELU(),
            nn.AdaptiveAvgPool2d((8, 27)),
        )
        ast = ASTForAudioClassification.from_pretrained(
            AST_HF, ignore_mismatched_sizes=True)
        self.transformer = ast.audio_spectrogram_transformer.encoder
        self.layernorm   = ast.audio_spectrogram_transformer.layernorm

        self.cls_token = nn.Parameter(torch.zeros(1, 1, 768))
        self.pos_embed = nn.Parameter(torch.zeros(1, 217, 768))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.head = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(768, num_classes)
        )

    def forward(self, x):
        feats  = self.cnn_stem(x)
        B      = feats.shape[0]
        tokens = feats.flatten(2).transpose(1, 2)
        cls    = self.cls_token.expand(B, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1) + self.pos_embed
        out    = self.transformer(tokens).last_hidden_state
        out    = self.layernorm(out)
        return self.head(out[:, 0])


# AST training
train_ld_ast, val_ld_ast = build_loaders_from_split(
    TRAIN_FILES, VAL_FILES, batch_train=BATCH_AST, batch_val=64)

ast_model     = ASTGenreClassifier().to(DEVICE)
pretrained_params = (list(ast_model.transformer.parameters()) +
                     list(ast_model.layernorm.parameters()))
pretrained_ids    = {id(p) for p in pretrained_params}
new_params        = [p for p in ast_model.parameters()
                     if id(p) not in pretrained_ids]

# UPGRADED: lower transformer LR (1e-5 → 5e-6) for better fine-tuning
optimizer_ast = torch.optim.AdamW([
    {'params': pretrained_params, 'lr': 5e-6},
    {'params': new_params,        'lr': 3e-4},
], weight_decay=1e-4)
criterion_ast = nn.CrossEntropyLoss(label_smoothing=0.1)
scheduler_ast = get_scheduler(optimizer_ast, warmup_epochs=2, total_epochs=AST_EPOCHS)
scaler_ast    = torch.cuda.amp.GradScaler() if DEVICE.type == 'cuda' else None

best_f1_a      = 0.0
oof_logits_ast = None
oof_labels_ast = None

for epoch in range(1, AST_EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(ast_model, train_ld_ast, optimizer_ast,
                                      criterion_ast, scaler_ast)
    val_f1, val_logits, val_labels = evaluate(ast_model, val_ld_ast)
    scheduler_ast.step()

    marker = ''
    if val_f1 > best_f1_a:
        best_f1_a      = val_f1
        oof_logits_ast = val_logits
        oof_labels_ast = val_labels
        torch.save({'model_state': ast_model.state_dict(), 'val_f1': val_f1},
                   f'{CKPT_DIR}/best_ast.pt')
        marker = '  ← best'

    print(f'[AST] Ep {epoch:02d}/{AST_EPOCHS}  '
          f'loss={tr_loss:.4f}  acc={tr_acc:.4f}  '
          f'val_f1={val_f1:.4f}{marker}')

print(f'\n✅ AST best val macro-F1: {best_f1_a:.4f}')

In [ ]:
# ── CatBoost Stacker — 3-model ensemble (ResNet50 + EfficientNet + AST) ────
from catboost import CatBoostClassifier
import torch.nn.functional as F
from sklearn.model_selection import train_test_split


def to_probs(logits_tensor):
    return F.softmax(logits_tensor, dim=-1).numpy()


n = min(len(oof_logits_resnet), len(oof_logits_effnet), len(oof_logits_ast))

# Stack all 3 model probs: (n, 30)
X_oof = np.hstack([
    to_probs(oof_logits_resnet[:n]),   # (n, 10)
    to_probs(oof_logits_effnet[:n]),   # (n, 10)
    to_probs(oof_logits_ast[:n]),      # (n, 10)
])
y_oof = oof_labels_resnet[:n]

X_cb_train, X_cb_val, y_cb_train, y_cb_val = train_test_split(
    X_oof, y_oof, test_size=0.2, random_state=SEED, stratify=y_oof)

print(f'CatBoost stacker → train: {len(X_cb_train)}  val: {len(X_cb_val)}')
print(f'Input features: {X_oof.shape[1]} (10 probs × 3 models)')

cb = CatBoostClassifier(
    iterations=1000,        # ↑ was 500
    learning_rate=0.03,     # ↓ slower = better generalisation
    depth=6,
    loss_function='MultiClass',
    eval_metric='TotalF1',
    random_seed=SEED,
    verbose=100,
    early_stopping_rounds=80,
    l2_leaf_reg=3.0,        # NEW: regularisation
)
cb.fit(X_cb_train, y_cb_train, eval_set=(X_cb_val, y_cb_val))

cb_val_pred = cb.predict(X_cb_val).flatten().astype(int)
stack_f1    = f1_score(y_cb_val, cb_val_pred, average='macro')
print(f'\n✅ CatBoost stacker held-out macro-F1: {stack_f1:.4f}')

# ── Also learn optimal linear blend weights ──────────────────────────────
# Simple: val F1-weighted blend (used as fallback if CatBoost fails)
total_f1 = best_f1_r + best_f1_e + best_f1_a
W_RESNET  = best_f1_r / total_f1
W_EFFNET  = best_f1_e / total_f1
W_AST     = best_f1_a / total_f1
print(f'\nLinear blend weights (val F1-weighted):')
print(f'  ResNet50     : {W_RESNET:.3f}')
print(f'  EfficientNet : {W_EFFNET:.3f}')
print(f'  AST          : {W_AST:.3f}')

In [ ]:
# Load best checkpoints
ckpt_r = torch.load(f'{CKPT_DIR}/best_resnet50.pt', map_location=DEVICE)
resnet.load_state_dict(ckpt_r['model_state'])
resnet.eval()
print(f'ResNet50     loaded  (val_f1={ckpt_r["val_f1"]:.4f})')

ckpt_e = torch.load(f'{CKPT_DIR}/best_effnet_b2.pt', map_location=DEVICE)
effnet.load_state_dict(ckpt_e['model_state'])
effnet.eval()
print(f'EfficientNet loaded  (val_f1={ckpt_e["val_f1"]:.4f})')

ckpt_a = torch.load(f'{CKPT_DIR}/best_ast.pt', map_location=DEVICE)
ast_model.load_state_dict(ckpt_a['model_state'])
ast_model.eval()
print(f'AST          loaded  (val_f1={ckpt_a["val_f1"]:.4f})')

In [ ]:
@torch.no_grad()
def tta_logits(model, mel_1chw: torch.Tensor, n_tta=N_TTA):
    """
    UPGRADED TTA: stronger spec-augment + random time shift.
    mel_1chw : (1, 1, 128, 431) on CPU
    Returns  : np.ndarray (NUM_CLASSES,)
    """
    model.eval()
    total = None

    # View 1: no augmentation (clean prediction)
    inp = mel_1chw.to(DEVICE)
    out = model(inp).squeeze(0).cpu().numpy()
    total = out

    for _ in range(n_tta - 1):
        aug = mel_1chw.clone().squeeze(0)  # (1,128,431)
        _, F, T = aug.shape

        # Frequency masking
        for _ in range(2):
            f  = random.randint(0, 20)
            f0 = random.randint(0, max(0, F - f - 1))
            aug[:, f0:f0+f, :] = 0

        # Time masking
        for _ in range(2):
            t  = random.randint(0, 50)
            t0 = random.randint(0, max(0, T - t - 1))
            aug[:, :, t0:t0+t] = 0

        # NEW: random time shift (circular roll)
        if random.random() < 0.5:
            shift = random.randint(-T//8, T//8)
            aug = torch.roll(aug, shifts=shift, dims=2)

        inp = aug.unsqueeze(0).to(DEVICE)
        out = model(inp).squeeze(0).cpu().numpy()
        total = total + out

    return total / n_tta


def softmax_np(x):
    e = np.exp(x - x.max())
    return e / e.sum()


print(f'✅ TTA inference helper ready  (N_TTA={N_TTA})')

In [ ]:
import pandas as pd
from pathlib import Path

test_df     = pd.read_csv(TEST_CSV)
mashups_dir = Path(MASHUPS_DIR)

print('test.csv columns:', test_df.columns.tolist())
print(test_df.head())
print(f'Total rows: {len(test_df)}')

all_mashup_files = list(mashups_dir.glob('*'))
print(f'\nFiles in {MASHUPS_DIR}: {len(all_mashup_files)}')
print('Sample:', [f.name for f in all_mashup_files[:5]])


def resolve_audio_path(row, mashups_dir):
    candidates = []
    for col in ['filename', 'file', 'audio', 'path', 'audio_path']:
        if col in row and pd.notna(row[col]):
            candidates.append(mashups_dir / row[col])
            candidates.append(mashups_dir / Path(row[col]).name)

    sample_id = str(row['id'])
    for ext in ['.wav', '.mp3', '.ogg', '.flac']:
        candidates += [
            mashups_dir / f'{sample_id}{ext}',
            mashups_dir / f'mashup_{sample_id}{ext}',
            mashups_dir / f'{sample_id.zfill(5)}{ext}',
        ]
        candidates += list(mashups_dir.glob(f'*{sample_id}*{ext}'))

    for p in candidates:
        if Path(p).exists():
            return str(p)
    return None


predictions = []
found_count  = 0
missed_count = 0

print(f'\nRunning inference on {len(test_df)} test samples...')

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Predicting'):
    sample_id  = row['id']
    audio_path = resolve_audio_path(row, mashups_dir)

    if audio_path is None:
        missed_count += 1
        y          = np.zeros(int(SAMPLE_RATE * CLIP_DURATION), dtype=np.float32)
        mel_tensor = audio_to_mel(y).unsqueeze(0)
        logits_r   = tta_logits(resnet, mel_tensor, n_tta=1)
        genre_idx  = int(np.argmax(logits_r))
        predictions.append({'id': sample_id, 'label': IDX_TO_GENRE[genre_idx]})
        continue

    found_count += 1
    try:
        y = load_audio(audio_path, duration=CLIP_DURATION)
        if len(y) < int(SAMPLE_RATE * CLIP_DURATION):
            y = np.pad(y, (0, int(SAMPLE_RATE * CLIP_DURATION) - len(y)))

        # Use global normalization at inference time (NEW)
        mel_tensor = audio_to_mel(y, global_norm=True).unsqueeze(0)  # (1,1,128,431)

        # TTA from all 3 models
        logits_r = tta_logits(resnet,    mel_tensor)
        logits_e = tta_logits(effnet,    mel_tensor)
        logits_a = tta_logits(ast_model, mel_tensor)

        # CatBoost stacker (primary)
        probs     = np.hstack([softmax_np(logits_r),
                               softmax_np(logits_e),
                               softmax_np(logits_a)])
        genre_idx = int(cb.predict([probs])[0])

    except Exception as e:
        print(f'  [WARN] {sample_id}: {e}')
        try:
            # Weighted blend fallback
            p_r = softmax_np(logits_r) if 'logits_r' in dir() else np.ones(NUM_CLASSES)/NUM_CLASSES
            p_e = softmax_np(logits_e) if 'logits_e' in dir() else p_r
            p_a = softmax_np(logits_a) if 'logits_a' in dir() else p_r
            blend     = W_RESNET * p_r + W_EFFNET * p_e + W_AST * p_a
            genre_idx = int(np.argmax(blend))
        except Exception:
            genre_idx = random.randint(0, NUM_CLASSES - 1)

    predictions.append({'id': sample_id, 'label': IDX_TO_GENRE[genre_idx]})


print(f'\n✅ Found audio for {found_count}/{len(test_df)} samples ({missed_count} missed)')

sub_df = pd.DataFrame(predictions)
sub_df.to_csv(SUBMISSION, index=False)
print(f'✅ Saved {SUBMISSION}')
print('\nClass distribution in submission:')
print(sub_df['label'].value_counts())

In [ ]:
# ── Submission validation ─────────────────────────────────────────────────
sub          = pd.read_csv(SUBMISSION)
valid_genres = set(GENRES)

assert 'id'    in sub.columns
assert 'label' in sub.columns
assert len(sub) == len(test_df)

invalid = sub[~sub['label'].isin(valid_genres)]
assert len(invalid) == 0

vc = sub['label'].value_counts()
assert len(vc) > 1

print('✅ Submission format valid!')
print(f'   Rows    : {len(sub)}')
print(f'   Columns : {sub.columns.tolist()}')
print(f'\nDistribution:')
print(vc)

In [ ]:
print('=' * 60)
print('        MESSY MASHUP V2 — PIPELINE SUMMARY')
print('=' * 60)
print(f'  Training samples generated : {NUM_SAMPLES:,}')
print(f'  ResNet50 best val F1        : {best_f1_r:.4f}')
print(f'  EfficientNet-B2 best val F1 : {best_f1_e:.4f}')
print(f'  AST best val F1             : {best_f1_a:.4f}')
print(f'  CatBoost held-out F1        : {stack_f1:.4f}')
print(f'  TTA views                   : {N_TTA}')
print(f'  Submission file             : {SUBMISSION}')
print(f'  Total test predictions      : {len(sub_df)}')
print(f'  Files found / total         : {found_count} / {len(test_df)}')
print('=' * 60)
print()
print('UPGRADES vs V1:')
print('  1. ResNet50 instead of ResNet18 (deeper features)')
print('  2. EfficientNet-B2 added as 3rd model')
print('  3. Pitch-shift augmentation during data synthesis')
print('  4. Harder SNR range: 3-25 dB (was 5-30)')
print('  5. Mixup training for better boundary generalisation')
print('  6. Warmup LR scheduler for all models')
print('  7. Global mel stats normalisation at inference')
print('  8. N_TTA=10 with time-shift augmentation')
print('  9. CatBoost now stacks 3 models (30 features, was 20)')
print(' 10. 30k training samples (was 25k)')

In [ ]:
# # ── Ensemble template (run after training a second model) ─────────────────────
# # Uncomment and fill in model2 path when ready.

# model2 = ASTForAudioClassification.from_pretrained(
#     MODEL_NAME, num_labels=len(GENRES), ignore_mismatched_sizes=True
# )
# model2.load_state_dict(torch.load('/kaggle/working/ast_best_run2.pt', map_location=device))
# model2.to(device)

# @torch.no_grad()
# def predict_ensemble(models, audio, weights=None, n_crops=TTA_CROPS):
#     if weights is None:
#         weights = [1.0 / len(models)] * len(models)
#     combined = None
#     for m, w in zip(models, weights):
#         m.eval()
#         logits_list = []
#         for _ in range(n_crops):
#             start = random.randint(0, max(0, len(audio) - CROP_SAMPLES))
#             crop = audio[start:start + CROP_SAMPLES]
#             inputs = feature_extractor(crop, sampling_rate=SR, return_tensors='pt')
#             x = inputs['input_values'].to(device)
#             logits_list.append(m(x).logits.cpu())
#         avg = torch.mean(torch.stack(logits_list), dim=0)
#         combined = w * avg if combined is None else combined + w * avg
#     return torch.argmax(combined, dim=1).item()

# # Re-run inference with ensemble
# all_ids, all_preds = [], []
# for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Ensemble inference'):
#     audio, _ = librosa.load(os.path.join(MASHUP_DIR, row['filename']), sr=SR)
#     pred_idx = predict_ensemble([model, model2], audio, weights=[0.55, 0.45])
#     all_ids.append(int(row['id']))
#     all_preds.append(GENRES[pred_idx])
# submission = pd.DataFrame({'id': all_ids, 'genre': all_preds})
# submission.to_csv(os.path.join(OUTPUT_DIR, 'submission_ensemble.csv'), index=False)
# print('Ensemble submission saved.')

# print('Ensemble template ready — uncomment when you have a second model checkpoint.')